# Extract features from dataset.csv + checkpoint.json

This notebook builds the featured dataset from the two files produced by the
pipeline run: `dataset.csv` (already has `gold_answer`, `solver_correct`,
`revised_correct`, and every other parsed/derived column from `verify.py`)
and `checkpoint.json` (the only source of `critique_raw`, the full critique
text, which `dataset.csv` doesn't carry).

This is the direct path — use this whenever both files are available. It
does **not** need a separate gold-answer lookup: `dataset.csv` already has
`gold_answer`, so nothing needs to be reconstructed.

`features.py` itself now handles recovering `critique_raw` from
`checkpoint.json` and joining it on (`attach_critique_raw`), so this
notebook only needs `features.py` — no separate merge script.

(If you only have `checkpoint.json`, with no `dataset.csv` at all, use
`TryFeaturesFromCheckpoint.ipynb` instead — that one reconstructs
everything, including gold answers, from checkpoint alone.)

## 1. Set paths

In [1]:
import os
import sys

BASE_DIR = "/kaggle/input/datasets/ghassanbinhadi/datav6/v6_final_test_150_reparsed"

DATASET_PATH = f"{BASE_DIR}/data/processed/dataset_reparsed.csv"
OUTPUT_PATH = "/kaggle/working/test_dataset_with_features.csv"

# Concatenate the interaction JSONL files into a single temporary checkpoint file
REPARSED_DIR = f"{BASE_DIR}/data/reparsed"
COMBINED_CHECKPOINT = "/kaggle/working/combined_test_interactions.jsonl"

with open(COMBINED_CHECKPOINT, "w", encoding="utf-8") as outfile:
    for fname in sorted(os.listdir(REPARSED_DIR)):
        if fname.endswith(".jsonl"):
            with open(os.path.join(REPARSED_DIR, fname), "r", encoding="utf-8") as infile:
                for line in infile:
                    if line.strip():
                        outfile.write(line.strip() + "\n")

print(f"Combined pipeline JSONL prepared at: {COMBINED_CHECKPOINT}")

# Add BASE_DIR to import features.py from datav6
sys.path.insert(0, BASE_DIR)

Combined pipeline JSONL prepared at: /kaggle/working/combined_test_interactions.jsonl


## 2. Load dataset.csv and recover critique_raw from checkpoint.json

`attach_critique_raw()` (in `features.py`) does the join and prints a
join-quality summary — flags any row that has a `critic_verdict` but no
`critique_raw`, which would indicate a real join miss rather than a critique
stage that simply never ran.

In [2]:
import pandas as pd
import features as feat

dataset = pd.read_csv(DATASET_PATH)
print(f"Loaded {DATASET_PATH}: {dataset.shape[0]} rows, {dataset.shape[1]} cols")
print("gold_answer already present:", "gold_answer" in dataset.columns)

# Join raw critique text from the combined interaction records
dataset = feat.attach_critique_raw(dataset, COMBINED_CHECKPOINT)

Loaded /kaggle/input/datasets/ghassanbinhadi/datav6/v6_final_test_150_reparsed/data/processed/dataset_reparsed.csv: 300 rows, 42 cols
gold_answer already present: True
[features] Joined critique_raw for 298/300 rows (2 unmatched).


[features] NOTE: 2 records had no 'critique' stage at all (critique never ran for them).


## 3. Apply features.py

Same `build_features()` used everywhere else in this project — no
duplicated feature logic.

In [3]:
featured = feat.build_features(dataset)

direction_dummies = feat.direction_dummy_columns(featured)
candidate_features = feat.feature_columns(
    direction_dummies=direction_dummies,
    force_include=["proposed_answer_differs"],  # kept visible per project decision, not silently dropped
)

audit = feat.audit_features(candidate_features)
assert audit["ok"], f"Feature boundary violation: {audit}"

mask = feat.disagreement_population_mask(featured)
variance_report = feat.variance_check(featured, candidate_features, mask)
print(feat.format_variance_report(variance_report))

active_features = variance_report["kept"]
print(f"\nActive (passed variance check): {active_features}")
print(f"Dropped: {variance_report['dropped']}")
print(f"\nCharacterization-only, in the dataset but not classifier-default: {feat.CHARACTERIZATION_ONLY_FEATURES}")


Variance check on disagreement population (n=37, near_constant_frac=0.05)
feature                     nuniq  minority  decision reason
------------------------------------------------------------------------
answer_distance_log1p          19    0.8919  keep     ok
answer_distance_norm           21    0.8919  keep     ok
answer_distance_signed_log1p     33    0.9459  keep     ok
hedging_count                   3    0.5135  keep     ok
error_desc_length              25    0.8919  keep     ok
proposed_answer_differs         2    0.0270  DROP     near_constant(minority=0.0270<0.05)
direction_llama_solver_qwen_critic      2    0.4865  keep     ok
direction_qwen_solver_llama_critic      2    0.4865  keep     ok
------------------------------------------------------------------------
kept:    ['answer_distance_log1p', 'answer_distance_norm', 'answer_distance_signed_log1p', 'hedging_count', 'error_desc_length', 'direction_llama_solver_qwen_critic', 'direction_qwen_solver_llama_critic']
dropped

## 4. Save the featured dataset

In [4]:
featured.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {OUTPUT_PATH}: {featured.shape[0]} rows, {featured.shape[1]} cols")


Wrote /kaggle/working/test_dataset_with_features.csv: 300 rows, 52 cols


## 5. Iterate here

`predictor_df` is the disagreement-only population
(`valid == 1 & revision_called == 1`) used for modeling. From here it's the
same shape as `ModelingV2_FIXED.ipynb`.

In [5]:
predictor_df = featured[(featured["valid"] == 1) & (featured["revision_called"] == 1)].copy()
y = predictor_df["beneficial"]
families = feat.feature_families(active_features)
print("metadata features:", families["metadata"])
print("structured features:", families["structured"])
print("positive rate:", y.mean(), "n =", len(y))


metadata features: ['error_desc_length']
structured features: ['answer_distance_log1p', 'answer_distance_norm', 'answer_distance_signed_log1p', 'hedging_count', 'direction_llama_solver_qwen_critic', 'direction_qwen_solver_llama_critic']
positive rate: 0.13513513513513514 n = 37
